# After running this script you should at the end see the EURR of your methods
### Pay attention that you have the heuristics and strategies results in the EURR dir before running this notebook. In order to get those results, you need to run the evaluation.ipynb files in those dirs

In [1]:
import pandas as pd
LLMS = [
"EleutherAI-pythia-6.9b",
"meta-llama-Llama-3.1-8B",
"meta-llama-Meta-Llama-3-8B-Instruct"
]
strategies=["G-Greedy","G-Utility","Random"]
heuristics=["nb","maxsp","fair_round_robin","random"]
utilities=["perplexity","vc"]

In [2]:
strategies_dict={}
for llm in LLMS:
    for s in strategies:
        for u in utilities:
            strategies_dict[(llm,s,u)]=pd.read_parquet(f'strategy_total_results/series_{s}_{u}_{llm}.pq')
            strategies_dict[(llm,s,u)]=strategies_dict[(llm,s,u)].rename({strategies_dict[(llm,s,u)].columns[0]:'score'},axis=1)

In [ ]:
new_strategies_dict={}
for llm in LLMS:
    for s in strategies:
        new_strategies_dict[(llm,s)]=pd.DataFrame()
        new_strategies_dict[(llm,s)][utilities[0]]= strategies_dict[(llm,s,utilities[0]])['score']
        new_strategies_dict[(llm,s)][utilities[1]]= strategies_dict[(llm,s,utilities[1]])['score']

In [ ]:
new_new_strategies_dict={}
for llm in LLMS:
    new_new_strategies_dict[llm]=pd.DataFrame()
    for s in strategies:
        new_new_strategies_dict[llm][utilities[0]+"_"+s]= strategies_dict[(llm,s,utilities[0]])['score']
        new_new_strategies_dict[llm][utilities[1]+"_"+s]= strategies_dict[(llm,s,utilities[1]])['score']

In [ ]:
import pandas as pd

# 1. Define the filename
output_file = 'strategies_by_samples.xlsx'

# 2. Use ExcelWriter to save multiple sheets
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for sheet_name, df in new_new_strategies_dict.items():
        # Write each dataframe to a specific sheet
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"File saved as {output_file}")

In [ ]:
heuristics_dict={}
for llm in LLMS:
    for s in heuristics:
        for u in utilities:
            heuristics_dict[(llm,s,u)]=pd.read_parquet(f'total_results/series_{s}_{u}_{llm}.pq')
            heuristics_dict[(llm,s,u)]=heuristics_dict[(llm,s,u)].rename({heuristics_dict[(llm,s,u)].columns[0]:'score'},axis=1)

In [ ]:
new_new_heuristics_dict={}
old_heuristics = ["random", "nb", "maxsp", "fair_round_robin"]
new_heuristics = ["random", "GreedyNP", "MaxSP", "MPP"]
mapping = dict(zip(old_heuristics, new_heuristics))

for llm in LLMS:
    new_new_heuristics_dict[llm]=pd.DataFrame()
    for s in heuristics:
        new_new_heuristics_dict[llm][utilities[0]+"_"+s]= heuristics_dict[(llm,s,utilities[0]])['score']
        new_new_heuristics_dict[llm][utilities[1]+"_"+s]= heuristics_dict[(llm,s,utilities[1]])['score']

for llm in LLMS:
    for old_val, new_val in mapping.items():
    # regex=False ensures we treat the old_val as a literal string, not a pattern
        new_new_heuristics_dict[llm].columns = new_new_heuristics_dict[llm].columns.str.replace(old_val, new_val, regex=False)

In [ ]:

#mapping to fit the results in the article
new_heuristics=["random","GreedyNP","MaxSP","MPP"]
old_heuristics=["random","nb","maxsp","fair_round_robin"]

df=new_new_heuristics_dict[llm]
import pandas as pd

# 1. Setup your mapping
old_heuristics = ["random", "nb", "maxsp", "fair_round_robin"]
new_heuristics = ["random", "GreedyNP", "MaxSP", "MPP"]
mapping = dict(zip(old_heuristics, new_heuristics))

# 2. Iterate and replace the substring in the column names
for old_val, new_val in mapping.items():
    # regex=False ensures we treat the old_val as a literal string, not a pattern
    df.columns = df.columns.str.replace(old_val, new_val, regex=False)

# Check the results
print(df.columns)

In [ ]:
import pandas as pd

# 1. Define the filename
output_file = 'heuristics_by_samples.xlsx'

# 2. Use ExcelWriter to save multiple sheets
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for sheet_name, df in new_new_heuristics_dict.items():
        # Write each dataframe to a specific sheet
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"File saved as {output_file}")

In [ ]:
import pandas as pd
import numpy as np

def calculate_eurr(strategies_dict, heuristics_dict, score_col='score'):
    results = []

    # Iterate through every strategy configuration
    for (llm, strategy, utility), strat_df in strategies_dict.items():
        
        # 1. Identify "a" (The Strategy Scores)
        if score_col not in strat_df.columns:
            print(f"Skipping ({llm}, {strategy}, {utility}): Column '{score_col}' not found.")
            continue
        
        a_series = strat_df[score_col]
        
        # 2. Identify "b, c..." (The Heuristic Scores)
        # We need to find all heuristics that match the current LLM and Utility
        matched_heuristics = {}
        for (h_llm, h_name, h_utility), h_df in heuristics_dict.items():
            if h_llm == llm and h_utility == utility:
                if score_col in h_df.columns:
                    matched_heuristics[h_name] = h_df[score_col]
        
        if not matched_heuristics:
            print(f"Warning: No matching heuristics found for ({llm}, {strategy}, {utility})")
            continue

        # Create a generic DataFrame for Heuristics to allow easy row-wise operations
        # Index should align with a_series based on assumption
        heuristics_df = pd.DataFrame(matched_heuristics)
        

        # --- Method  Calculation ---
        # Formula: Max of ( Average(a/b), Average(a/c), ... )
        
        avg_ratios_per_heuristic = []
        
        for col in heuristics_df.columns:
            
            h_series = heuristics_df[col]
            # Calculate mean ratio for this specific heuristic
            current_heuristic_mean_ratio = (a_series / h_series).mean()
            avg_ratios_per_heuristic.append(current_heuristic_mean_ratio)
        method_2_score = min(avg_ratios_per_heuristic)
        
        # Store results
        results.append({
            'llm': llm,
            'strategy': strategy,
            'utility': utility,
            'eurr_method': method_2_score
        })

    return pd.DataFrame(results)

# --- Usage Example ---
# Assuming you have already populated strategies_dict and heuristics_dict
df_results = calculate_eurr(strategies_dict, heuristics_dict)
df_results